# Ice water content from a size distribution, and how much the mass relation matters

A size distribution counts particles; ice water content needs each particle's mass, which a probe
does not measure. The package keeps its mass-diameter relations in one registry,
`arrayprobe_enhancements.md_relations`, where each entry names its source, its validity range and
the code that applies it. This notebook selects the Brown and Francis (1995) power law from that
registry, integrates a seeded synthetic exponential size distribution into ice water content,
checks the result against the closed-form integral, and quotes how far a solid-ice-sphere
assumption would move the same number, in dB. It runs in seconds and needs no data files.

**Before this:** [06_read_products](../1_getting_started/06_read_products.ipynb) (Reading the output files). **Related:** [01_microphysics_moments](01_microphysics_moments.ipynb). **Needs:** the enhancements package.

This notebook reads files from the arrayprobe source repository, so run it from a clone of that repository; `pip install arrayprobe` alone does not include them.

In [1]:
# Launch guard: the cells below reach scripts/ and tests/data/ relative to the repository root.
from pathlib import Path

if not (Path("scripts").is_dir() and Path("tests/data").is_dir()):
    raise RuntimeError(
        f"This notebook must run with the repository root as its working directory, not {Path.cwd()}. "
        "Jupyter starts a kernel in the notebook's own folder (examples/6_science/), so run "
        "`%cd ../..` in a new cell, then run this cell again. "
        "Headless, from the repository root: python scripts/execute_example_notebooks.py --check <name>"
    )

## 1. Select the relation from the registry

`md_relations.available()` lists the registered relations; `get()` returns one with its form,
validity range and the function that applies it. The Brown and Francis law is
`m = a D^b` with `D` in microns and `m` in grams, valid for `D >= 100` microns; below that,
`mass_bf95` raises rather than switching to another assumption.

In [2]:
import math
import numpy as np
from arrayprobe_enhancements import md_relations
from arrayprobe_enhancements.brown_francis_1995 import A_COEFFICIENT_G_UM, B_EXPONENT, mass_bf95

print(md_relations.available())
rel = md_relations.get("brown_francis_1995")
print(rel.form, rel.diameter_range_mm, rel.implementation)
assert rel.implementation.endswith("mass_bf95")
assert math.isclose(mass_bf95(1000.0), A_COEFFICIENT_G_UM * 1000.0 ** B_EXPONENT, rel_tol=1e-12)
try:
    mass_bf95(50.0)
    raise AssertionError("expected a refusal below 100 microns")
except ValueError:
    print("50 microns refused: outside the relation's stated range")

['baker_lawson_2006_area', 'brown_francis_1995', 'doc0223_effective_density', 'uiops_single_mass']
power_law_diameter (0.1, None) arrayprobe_enhancements.brown_francis_1995.mass_bf95
50 microns refused: outside the relation's stated range


## 2. Ice water content of an exponential distribution

The synthetic distribution is `N(D) = N0 exp(-lambda D)` above 100 microns, sampled on 2,000
bins of 5 microns with seeded Poisson counting noise on a 10-litre sample. Ice water content is the
sum over bins of concentration times mass. Without noise the closed form is
`a N0 [Gamma(b+1, lambda D_min)] / lambda^(b+1)`, the upper incomplete gamma function, computed
here from its series so only numpy is needed; the binned sum must recover it within 0.1 %, and
the noisy sample within three standard deviations of its
Poisson counting uncertainty, computed from the same bins. Because the few largest particles carry
most of the mass, that uncertainty is about ten percent even with a few hundred particles counted.

In [3]:
N0 = 0.2             # particles per litre per micron
lam = 1.0 / 250.0    # per micron (mean diameter 250 microns above the cut)
edges = np.arange(100.0, 100.0 + 5.0 * 2001, 5.0)
mid, width = 0.5 * (edges[1:] + edges[:-1]), np.diff(edges)
conc = N0 * np.exp(-lam * mid) * width                       # particles per litre per bin
mass = np.array([mass_bf95(d) for d in mid])                  # grams
iwc = float(np.sum(conc * mass)) * 1e3                        # g per m^3 (1000 L per m^3)

s, x = B_EXPONENT + 1.0, lam * edges[0]
k = np.arange(200)
lower = x ** s * math.exp(-x) * np.sum(np.cumprod(x / (s + k)) / x) if x > 0 else 0.0
upper_gamma = math.gamma(s) - lower
iwc_exact = A_COEFFICIENT_G_UM * N0 * upper_gamma / lam ** s * 1e3
print(f"binned IWC {iwc:.5f} g m-3, closed form {iwc_exact:.5f} g m-3, "
      f"difference {100 * (iwc / iwc_exact - 1):.4f} %")
assert abs(iwc / iwc_exact - 1) < 1e-3

rng = np.random.default_rng(48)
V = 10.0                                                      # litres sampled
counts = rng.poisson(conc * V)
iwc_noisy = float(np.sum(counts * mass)) / V * 1e3
print(f"with counting noise: {iwc_noisy:.5f} g m-3 from {counts.sum()} particles "
      f"({100 * (iwc_noisy / iwc_exact - 1):+.2f} %)")
sigma = float(np.sqrt(np.sum(conc * V * mass ** 2))) / V * 1e3     # Poisson standard deviation of IWC
print(f"expected counting uncertainty: {sigma:.5f} g m-3 ({100 * sigma / iwc_exact:.1f} %); "
      f"the sample is {(iwc_noisy - iwc_exact) / sigma:+.2f} standard deviations off")
assert abs(iwc_noisy - iwc_exact) < 3 * sigma

binned IWC 0.24023 g m-3, closed form 0.24023 g m-3, difference 0.0004 %
with counting noise: 0.24766 g m-3 from 337 particles (+3.09 %)
expected counting uncertainty: 0.02508 g m-3 (10.4 %); the sample is +0.30 standard deviations off


## 3. The same distribution under a solid-sphere assumption

Treating every particle as a solid ice sphere (density 0.917 g cm$^{-3}$) gives mass `rho pi D^3 / 6`.
For aggregates above 100 microns that overstates the mass, increasingly with size. The ratio of
the two ice water contents is quoted in dB, `10 log10(IWC_sphere / IWC_BF95)`.

In [4]:
rho = 0.917e-12                                               # grams per cubic micron
mass_sphere = rho * np.pi * mid ** 3 / 6.0
iwc_sphere = float(np.sum(conc * mass_sphere)) * 1e3
ratio_db = 10 * np.log10(iwc_sphere / iwc)
print(f"solid spheres: {iwc_sphere:.4f} g m-3, {ratio_db:.2f} dB above the power law")
for d in (100.0, 300.0, 1000.0):
    print(f"  D = {d:5.0f} microns: sphere/power-law mass {10 * np.log10(rho * np.pi * d ** 3 / 6 / mass_bf95(d)):.2f} dB")
assert ratio_db > 0

solid spheres: 2.2489 g m-3, 9.71 dB above the power law
  D =   100 microns: sphere/power-law mass 0.13 dB
  D =   300 microns: sphere/power-law mass 5.38 dB
  D =  1000 microns: sphere/power-law mass 11.13 dB


## Summary

- `md_relations.get()` returns a relation with its source, validity range and the function that
  applies it, so the mass assumption behind an ice water content is named rather than implied.
- Integrated on fine bins, `mass_bf95` recovers the closed-form ice water content of an
  exponential distribution within 0.1 %, and a 10-litre sample with counting noise within three standard deviations of its
  counting uncertainty.
- The choice of mass relation moves ice water content by several dB for the same counts: state
  the relation with every ice water content you report.